In [12]:
import matplotlib.pyplot as plt
import numpy as np

import cmbo
from IPython.display import display

from plot_stacked import *
from score_individual_objects import *
from prob_match import *

from jax.scipy.stats import norm as jax_norm
from jax.scipy.integrate import trapezoid
from jax.scipy.special import erf

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Sky fraction test

In [ ]:
@jax.jit
def f_sky(b_deg, b_cut_deg, sigma_theta_rad):
    """
    Compute sky survival fraction for Zone of Avoidance.
    """
    b_rad = jnp.deg2rad(jnp.abs(b_deg))
    b_cut_rad = jnp.deg2rad(b_cut_deg)
    return jax_norm.cdf((b_rad - b_cut_rad) / sigma_theta_rad)

def f_sky_numerical(b_deg, b_cut_deg, sigma_theta_rad):
    b_rad = jnp.deg2rad(b_deg)
    b_cut_rad = jnp.deg2rad(b_cut_deg)

    b_grid = jnp.linspace(-jnp.pi / 2, jnp.pi / 2, 10000)

    # Prior over b
    lp = 0.5 * jnp.cos(b_grid)

    # Gaussian in b, truncated but not periodic
    ll = jnp.exp(-0.5 * ((b_grid - b_rad) / sigma_theta_rad) ** 2)
    ll /= trapezoid(ll, x=b_grid)

    y = ll * jnp.where(jnp.abs(b_grid) > b_cut_rad, 1., 0.) * lp
    return trapezoid(y, x=b_grid)


from jax.scipy.special import erf, erfc

@jax.jit
def f_sky_latitude(b_rad, b_cut_rad, sigma_theta_rad):
    """
    Analytic f = E[1_{|b'|>b_cut}] with b' ~ N(b, sigma) on latitude and
    prior 0.5 cos(b').

    Numerically stable implementation using logsumexp trick for exponential
    terms.
    """
    # Precompute common subexpressions
    mu = b_rad
    b = b_cut_rad
    sigma2 = sigma_theta_rad**2
    inv_sigma2 = 1.0 / sigma2

    sqrt2_sigma = jnp.sqrt(2) * sigma_theta_rad
    pi_minus_2mu = jnp.pi - 2*mu
    pi_plus_2mu = jnp.pi + 2*mu
    b_minus_mu = b - mu
    b_plus_mu = b + mu

    # Erf terms
    erf_term1 = erf(pi_minus_2mu / (2 * sqrt2_sigma))
    erf_term2 = erf(pi_plus_2mu / (2 * sqrt2_sigma))
    erfc_term1 = 1 - erf(b_minus_mu / sqrt2_sigma)
    erfc_term2 = 1 - erf(b_plus_mu / sqrt2_sigma)

    # Trigonometric terms
    cos_mu = jnp.cos(mu)
    sin_mu = jnp.sin(mu)

    # First part of numerator
    erf_sum = -2 + erf_term1 + erf_term2 + erfc_term1 + erfc_term2
    numerator_part1 = cos_mu * erf_sum

    # Second part of numerator - exponential terms with numerical stability
    # Compute log of each exponential (the exponents)
    log_exp1 = -pi_minus_2mu**2 * 0.125 * inv_sigma2
    log_exp2 = -b_minus_mu**2 * 0.5 * inv_sigma2
    log_exp3 = -b_plus_mu**2 * 0.5 * inv_sigma2
    log_exp4 = -pi_plus_2mu**2 * 0.125 * inv_sigma2

    # Numerically stable computation via logsumexp trick
    # exp1 - exp2 + exp3 - exp4
    max_log = jnp.maximum(jnp.maximum(log_exp1, log_exp2),
                          jnp.maximum(log_exp3, log_exp4))

    exp_sum = (jnp.exp(log_exp1 - max_log)
               - jnp.exp(log_exp2 - max_log)
               + jnp.exp(log_exp3 - max_log)
               - jnp.exp(log_exp4 - max_log)) * jnp.exp(max_log)

    # Complete second part of numerator (precompute sqrt(2/pi))
    numerator_part2 = exp_sum * 0.7978845608028654 * sigma_theta_rad * sin_mu

    # Total and return
    return 0.5 * (numerator_part1 + numerator_part2) / (erf_term1 + erf_term2)


In [6]:
b_deg = -26.068897114676552
b_cut_deg = 5
sigma_theta_rad = jnp.deg2rad(3)


print(f_sky_numerical(b_deg, b_cut_deg, sigma_theta_rad))
print(f_sky_latitude(jnp.deg2rad(b_deg), jnp.deg2rad(b_cut_deg), sigma_theta_rad,))
# 

NameError: name 'f_sky_numerical' is not defined

### Redshift-part of the selection

In [10]:
def f_z_numerical(cz_max, cz_halo, sigma_v):
    cz_grid = jnp.linspace(-10 * sigma_v, cz_max + 10 * sigma_v, 10000)

    # Gaussian in cz_halo
    ll = jax_norm.pdf(cz_grid, loc=cz_halo, scale=sigma_v)

    y = ll * jnp.where(cz_grid <= cz_max, 1., 0.)
    return trapezoid(y, x=cz_grid)

@jax.jit
def f_z(cz_max, cz_halo, sigma_v):
    """Redshift-space selection function f_z."""
    return 0.5 * (1 + erf((cz_max - cz_halo) / (jnp.sqrt(2) * sigma_v)))

In [16]:
cz_max = 10_000
cz_halo = 9000

sigma_v = 3000


print(f_z_numerical(cz_max, cz_halo, sigma_v))
print(f_z(cz_max, cz_halo, sigma_v))

0.6303699621172318
0.6305586598182363
